# NF2 · Procesamiento y ecosistemas distribuidos (RA2)
## El motor predictivo de la *plataforma de juego online*

**Tu misión:** procesar **600.000 eventos de juego** con **Apache Spark** y
*demostrar* las propiedades que hacen viable el Big Data: poder de cómputo
distribuido, tolerancia a fallos, almacenamiento flexible y escalabilidad.

> Autoevaluable: la celda final genera `resultados.json`. Spark es determinista
> en los **conteos**; las **medias** pueden variar mínimamente (orden de suma
> distribuido), por eso se corrigen con tolerancia.

### Lo que demostrarás (criterios del RA2)
- **2.1** Procesar gran volumen rápidamente (carga distribuida).
- **2.2** Comprobar el poder de la computación distribuida (joins, agregaciones).
- **2.3** Probar la tolerancia a fallos (recomputación por **linaje**).
- **2.4** Almacenar y usar después (**schema-on-read**, partition pruning, streaming).
- **2.5** Ver que el sistema escala añadiendo módulos (particiones).

### Antes de empezar · genera los datos (una sola vez)

Abre una **terminal** (no una celda) y ejecuta, **desde la raíz del repositorio**:

```bash
cd nf2
python datos/generar_datos.py --salida datos/raw
```

Este cuaderno **se sitúa solo** en `nf2/`, así que todas sus rutas son relativas a esa
carpeta. Si la primera celda de código falla con un error de fichero no encontrado, es
que te falta este paso.


In [ ]:
import os
if os.path.basename(os.getcwd()) == "actividad": os.chdir("..")  # ejecutar desde la carpeta del núcleo (donde está datos/)
import glob
# --- Fuerza Java 17 para Spark (Java 18+ provoca el error getSubject) ---
_j = sorted(glob.glob("/usr/lib/jvm/*17*") + glob.glob("/usr/local/sdkman/candidates/java/17*"))
if _j:
    os.environ["JAVA_HOME"] = _j[0]; os.environ["PATH"] = _j[0] + "/bin:" + os.environ.get("PATH", "")
print("JAVA_HOME =", os.environ.get("JAVA_HOME", "(no encontrado: reconstruye el Codespace)"))
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

# Entorno (preinstalado en el devcontainer del curso)
import json, os, shutil
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, LongType, IntegerType,
                               StringType, DoubleType)

spark = (SparkSession.builder.appName("NF2")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
RAW, PROC, UMBRAL = "datos/raw", "datos/procesado", 90.0
resultados = {}
print("Spark", spark.version, "listo. Abre la Spark UI en el puerto 4040 para ver el DAG.")

---
## Fase 1 · Carga distribuida (RA2.1)
Carga `eventos.parquet` y la dimensión `regiones.csv` (con cabecera e inferencia de tipos).

In [ ]:
# Fase 1 · Carga distribuida
eventos = ...    # TODO: lee eventos.parquet con Spark · pista: spark.read.parquet(f"{RAW}/eventos.parquet")
regiones = ...   # TODO: lee regiones.csv con cabecera e inferSchema · pista: spark.read.option("header",True).option("inferSchema",True).csv(...)
resultados["fase1_carga"] = {
    "filas_total": int(eventos.count()),
    "num_columnas": len(eventos.columns),
    "columnas": sorted(eventos.columns),
}
resultados["fase1_carga"]

---
## Fase 2 · Poder de la computación distribuida (RA2.2)
1. **Transformación** (lazy) + **acción**: filtra `valor > UMBRAL` y cuenta los críticos.
2. **Join** `eventos × regiones` y agrega por `continente` → top-3 continentes por nº de eventos
   (ordena por count desc y, en empate, por `continente` asc → determinista).
3. Media de `valor` por `tipo`, en **dos pasos**: agregar en el clúster y traerse el
   resultado al *driver* con `.collect()` (una **acción**). Redondea a 2 decimales.

In [ ]:
# Fase 2 · Procesamiento distribuido (join + agregación)
criticos = ...        # TODO: filtra eventos con valor > UMBRAL · pista: eventos.filter(F.col("valor") > UMBRAL)
n_criticos = ...      # TODO: cuéntalos · pista: .count()
joined = ...          # TODO: une eventos con regiones por id_region (inner) trayendo "continente"
por_continente = ...  # TODO: agrupa el join por continente y cuenta, ordenando desc · pista: groupBy().count().orderBy(...)
top3 = ...            # TODO: los 3 continentes con más eventos · pista: [int(r["continente"]) for r in por_continente.take(3)]

# La media por tipo va en DOS pasos, como arriba (por_continente -> top3):
# primero agregas en el clúster, y después te traes el resultado al driver.
avg_res = ...         # TODO: media de valor por tipo, ya traída al driver.
                      #   OJO: .collect() es una ACCIÓN — es lo que dispara el cálculo (§2.1).
                      #   Ponle nombre a la columna con .alias(), o se llamará "avg(valor)".
                      #   pista: eventos.groupBy("tipo").agg(F.avg("valor").alias("media_valor")).collect()
avg_tipo = ...        # TODO: convierte avg_res en un dict {tipo: media} redondeado a 2 decimales
                      #   pista: {r["tipo"]: round(r["media_valor"], 2) for r in avg_res}

resultados["fase2_distribuido"] = {
    "n_eventos_criticos": int(n_criticos),
    "top3_grupos_por_eventos": top3,
    "avg_valor_por_tipo": {k: avg_tipo[k] for k in sorted(avg_tipo)},
}
resultados["fase2_distribuido"]

---
## Fase 3 · Tolerancia a fallos por linaje (RA2.3)
Cachea un DataFrame derivado y suma; **libera la caché** (`unpersist`) y vuelve a
sumar: Spark **recomputa** desde el linaje. Verifica que el total no cambia y
registra la profundidad del linaje (`rdd.toDebugString`).

In [ ]:
# Fase 3 · Tolerancia a fallos (linaje / recomputación)
derivado = joined.groupBy("tipo").count()
derivado.cache(); total_antes = derivado.agg(F.sum("count")).collect()[0][0]
derivado.unpersist()   # "perdemos" la cache: Spark debe recomputar desde el linaje
total_despues = ...    # TODO: vuelve a sumar el total tras el unpersist · pista: misma agregación que total_antes
profundidad = ...      # TODO: nº de pasos del linaje · pista: derivado.rdd.toDebugString().decode().count(chr(10))
resultados["fase3_tolerancia"] = {
    "recomputacion_correcta": bool(total_antes == total_despues),
    "total_eventos_join": int(total_antes),
    "profundidad_linaje": int(profundidad),
}
resultados["fase3_tolerancia"]

---
## Fase 4 · Almacenar y usar después — schema-on-read (RA2.4)
Escribe los eventos **particionados por `tipo`** en Parquet y luego lee **solo**
el tipo `exploracion` (partition pruning). Cuenta sus filas.

In [ ]:
# Fase 4 · Schema-on-read (particionado + pruning)
# TODO 1: escribe eventos particionado por "tipo" · pista: eventos.write.mode("overwrite").partitionBy("tipo").parquet(f"{PROC}/eventos_part")
# TODO 2: lee SOLO la partición tipo=="exploracion" · pista: spark.read.parquet(...).filter(F.col("tipo")=="exploracion")
filas_exploracion = ...   # TODO: cuenta las filas de esa partición
resultados["fase4_schema_on_read"] = {"filas_tipo_filtrado": int(filas_exploracion), "particionado_por": "tipo"}
resultados["fase4_schema_on_read"]

---
## Fase 5 · Escalabilidad — invariante al nº de particiones (RA2.5)
Calcula el conteo por `tipo` con `repartition(4)` y con `repartition(16)`: el
resultado debe ser **idéntico** (escalar añadiendo módulos no cambia el resultado).

In [ ]:
# Fase 5 · Escalabilidad (resultado invariante al nº de particiones)
def conteo_por_tipo(df):
    return {r["tipo"]: int(r["count"]) for r in df.groupBy("tipo").count().collect()}
r4 = ...    # TODO: conteo_por_tipo de eventos con 4 particiones · pista: eventos.repartition(4)
r16 = ...   # TODO: lo mismo con 16 particiones
resultados["fase5_escalabilidad"] = {
    "resultado_invariante_4_vs_16": bool(r4 == r16),
    "particiones_probadas": [4, 16],
    "conteo_por_tipo": {k: r4[k] for k in sorted(r4)},
}
resultados["fase5_escalabilidad"]

---
## Fase 6 · Streaming (Structured Streaming, RA2.4)
Lee `datos/raw/stream_in/` como **stream de ficheros**, agrega el conteo por `tipo`,
escribe a un *sink* en memoria con `trigger(availableNow=True)` (procesa todo lo
disponible y termina) y consulta el total.

In [ ]:
# Fase 6 · Structured Streaming (micro-batch sobre ficheros)
esquema = StructType([
    StructField("id_evento", LongType()), StructField("id_region", IntegerType()),
    StructField("tipo", StringType()), StructField("valor", DoubleType())])
# TODO 1: lee el stream de f"{RAW}/stream_in" con ese esquema · pista: spark.readStream.schema(esquema).parquet(...)
# TODO 2: agrupa por tipo y cuenta, escribiendo a memoria con trigger(availableNow=True) y outputMode("complete")
#         pista: .writeStream.format("memory").queryName("stream_counts")...start(); luego q.awaitTermination()
# TODO 3: lee los resultados · pista: spark.sql("SELECT tipo, count FROM stream_counts").collect()  -> guárdalo en 'filas'
total_stream = ...      # TODO: suma de todos los counts del stream
top_tipo_stream = ...   # TODO: el tipo con más eventos en el stream
resultados["fase6_streaming"] = {"total_eventos_stream": int(total_stream), "top_tipo_stream": top_tipo_stream}
resultados["fase6_streaming"]

### Visualización + Spark UI (C4)
Dibuja un gráfico de barras del conteo por tipo (o top-3 continentes) **y** abre la
Spark UI (puerto **4040**), localiza el job de la Fase 2 y comenta brevemente sus
**etapas (stages)** y dónde ocurre el **shuffle**.

In [ ]:
# C4 · Gráfico para comunicar (RA2)
import matplotlib.pyplot as plt
# TODO: dibuja un gráfico de barras de eventos por tipo (usa r4) · pista: plt.bar(list(r4.keys()), list(r4.values()))

---
## Fase 7 · Razonamiento (formato examen, RA2)
Responde en **máximo 12 líneas**, con terminología precisa:

**a)** Diferencia **escalado vertical (scale-up)** y **horizontal (scale-out)** e
indica cuál es el pilar de Spark y por qué (relación con la Fase 5).

**b)** Define **transformación** vs **acción** con un ejemplo de cada una (de tu
notebook) y explica qué es la **evaluación perezosa** y cómo la aprovecha el
optimizador (Catalyst).

**c)** A partir de la Fase 3, explica cómo logra Spark la **tolerancia a fallos**
sin replicar datos en memoria. ¿Cuándo usarías **batch** y cuándo **streaming**?

*(Escribe tu respuesta aquí.)*


---
## Celda final · Generar `resultados.json` (no modificar)

In [ ]:
ALUMNO = "TU_NOMBRE_Y_APELLIDOS"   # <-- pon aquí "Apellidos, Nombre"
resultados["metadata"] = {"caso":"gaming","seed":42,"alumno":ALUMNO}
assert ALUMNO != "TU_NOMBRE_Y_APELLIDOS", "⚠️ Pon tus Apellidos, Nombre en ALUMNO antes de entregar."
assert set(resultados) >= {"fase1_carga","fase2_distribuido","fase3_tolerancia",
                           "fase4_schema_on_read","fase5_escalabilidad","fase6_streaming"}, "Faltan fases"
json.dump(resultados, open("resultados.json","w",encoding="utf-8"), ensure_ascii=False, indent=2)
spark.stop()
print("✅ resultados.json generado. Entrega el .ipynb (y resultados.json).")

---
### Checkpoint de preparación al examen (no evaluable)
1. ¿Por qué `count()` dispara el cómputo y `filter()` no?
2. Tienes 4 nodos y añades 4 más: ¿qué tipo de escalado es y qué espera Spark?
3. Si se cae un nodo a mitad de un job, ¿cómo recupera Spark las particiones perdidas?